# 04 — Governance Report
**Model Risk Governance Toolkit | Lending Club Credit Risk**

This notebook:
1. Loads all results from notebooks 02 and 03
2. Generates Evidently monitoring dashboards
3. Renders the bank-style HTML validation report via Jinja2

> **Learning annotation — What goes into a bank validation report:**
> A real SR 11-7 validation report is a formal document submitted to a model risk
> committee (MRC) — typically a senior risk committee that approves models for
> production use. The report must be reproducible, auditable, and self-contained:
> a reviewer who wasn't involved in building the model should be able to read the
> report and understand the model's purpose, performance, limitations, and the
> validator's recommendation without needing to run any code.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path

from toolkit.report import render_report
from monitoring.evidently_dashboard import run_all as run_evidently

print('Libraries loaded.')

[evidently] WARNING: evidently not installed. Run: pip install evidently
Libraries loaded.


## 1. Load All Results

In [2]:
with open('../data/processed/validation_results.json') as f:
    val_results = json.load(f)

psi_table = pd.read_csv('../data/processed/psi_table.csv')
ks_table  = pd.read_csv('../data/processed/ks_table.csv')

train_df   = pd.read_parquet('../data/processed/train.parquet')
monitor_df = pd.read_parquet('../data/processed/monitor.parquet')

train_scores   = np.load('../data/processed/train_scores.npy')
monitor_scores = np.load('../data/processed/monitor_scores.npy')

# Re-hydrate fairness DataFrames from serialised dicts
fairness_for_report = {}
for key, val in val_results.get('fairness_serialised', {}).items():
    fairness_for_report[key] = {
        'n_flagged_80_rule': val['n_flagged_80_rule'],
        'approval_rates':    pd.DataFrame(val['approval_rates']),
        'equalized_odds':    pd.DataFrame(val['equalized_odds']),
        'predictive_parity': pd.DataFrame(val['predictive_parity']),
    }

print('All results loaded.')
print(f"Test AUC:     {val_results['test_metrics']['auc_roc']}")
print(f"Monitor AUC:  {val_results['monitor_metrics']['auc_roc']}")
print(f"ECE (after):  {val_results['ece_after']}")
print(f"PSI (scores): {val_results['prediction_psi']}")

All results loaded.
Test AUC:     0.6962
Monitor AUC:  0.7142
ECE (after):  0.0093
PSI (scores): 0.000920047701963857


## 2. Evidently Monitoring Dashboards

> **Learning annotation — Evidently under the hood:**
> Evidently applies a statistical test per column to detect drift:
> - **Numeric features**: Wasserstein distance (earth mover's distance) by default,
>   or KS test for small samples
> - **Categorical features**: Chi-squared test for distribution shift
> - **Model output (scores)**: Jensen-Shannon divergence or PSI
>
> Evidently's HTML dashboards are designed for non-technical stakeholders:
> risk managers, model sponsors, and regulators who need to understand drift
> without reading statistical output tables. A colour-coded summary (green/amber/red
> per feature) communicates the same information as the PSI table in notebook 03,
> but in a form accessible to a board-level risk committee.

In [3]:
from toolkit.preprocessing import NUMERIC_FEATURES, CATEGORICAL_FEATURES

feature_cols = (
    [c for c in NUMERIC_FEATURES    if c in train_df.columns] +
    [c for c in CATEGORICAL_FEATURES if c in train_df.columns]
)

try:
    run_evidently(
        train_df=train_df,
        monitor_df=monitor_df,
        train_scores=train_scores,
        monitor_scores=monitor_scores,
        feature_cols=feature_cols,
        target_col='default',
        output_dir='../reports/output'
    )
    print('Evidently dashboards generated in reports/output/')
    print('  Open reports/output/evidently_drift.html in a browser to view.')
    print('  Open reports/output/evidently_quality.html for data quality.')
    print('  Open reports/output/evidently_performance.html for performance comparison.')
except Exception as e:
    print(f'Evidently skipped: {e}')
    print('Install with: pip install evidently')

Evidently skipped: evidently is required. Install with: pip install evidently
Install with: pip install evidently


## 3. Assemble Results Dictionary for Report

In [4]:
report_results = {
    # Performance
    'train_metrics':        val_results['train_metrics'],
    'test_metrics':         val_results['test_metrics'],
    'monitor_metrics':      val_results['monitor_metrics'],

    # Calibration
    'ece_before':           val_results['ece_before'],
    'ece_after':            val_results['ece_after'],
    'calibration_monitor':  val_results['calibration_monitor'],

    # Drift
    'psi_table':            psi_table,
    'ks_table':             ks_table,
    'prediction_psi':       val_results['prediction_psi'],
    'target_drift':         val_results['target_drift'],
    'n_sig_psi':            val_results['n_sig_psi'],
    'n_moderate_psi':       val_results['n_moderate_psi'],
    'n_drifted_ks':         val_results['n_drifted_ks'],

    # Fairness (with DataFrames re-hydrated)
    'fairness':             fairness_for_report,

    # Threshold
    'threshold_candidates': val_results['threshold_candidates'],
}

print('Results dictionary assembled.')

Results dictionary assembled.


## 4. Render Validation Report

In [5]:
report_path = render_report(
    results=report_results,
    template_dir='../reports/templates',
    output_dir='../reports/output',
    processed_dir='../data/processed'
)

print(f'\nValidation report rendered: {report_path}')
print('Open the HTML file in a browser to view the full bank-style report.')

[report] Validation report saved to ..\reports\output\validation_report_2026-05-02.html

Validation report rendered: ..\reports\output\validation_report_2026-05-02.html
Open the HTML file in a browser to view the full bank-style report.


## 5. Report Summary

In [6]:
from toolkit.report import build_findings, _auto_recommendation

findings       = build_findings(report_results)
recommendation = _auto_recommendation(findings)

print('=' * 60)
print('VALIDATION REPORT SUMMARY')
print('=' * 60)
print(f'Model:           Logistic Regression Credit Default Model v1.0')
print(f'Test AUC:        {val_results["test_metrics"]["auc_roc"]}')
print(f'Test Gini:       {val_results["test_metrics"]["gini"]}')
print(f'Test KS:         {val_results["test_metrics"]["ks"]}')
print(f'Monitor AUC:     {val_results["monitor_metrics"]["auc_roc"]}')
print(f'ECE (after cal): {val_results["ece_after"]}')
print(f'Score PSI:       {val_results["prediction_psi"]}')
print(f'PSI Significant: {val_results["n_sig_psi"]} features')
print()
print(f'FINDINGS: {len(findings)}')
for f in findings:
    print(f'  [{f["risk_level"]:6s}] {f["title"]}')
print()
print(f'RECOMMENDATION: {recommendation}')
print('=' * 60)

VALIDATION REPORT SUMMARY
Model:           Logistic Regression Credit Default Model v1.0
Test AUC:        0.6962
Test Gini:       0.3925
Test KS:         0.2844
Monitor AUC:     0.7142
ECE (after cal): 0.0093
Score PSI:       0.000920047701963857
PSI Significant: 0 features

FINDINGS: 1
  [Low   ] 1 feature(s) with moderate PSI (0.10–0.25)

RECOMMENDATION: Approve
